# Faruq-v3 — STB capacity-causal control
Melatih hanya CMC0 seed 42 sebagai kontrol non-spatial yang parameter/depth/schedule-matched terhadap STB1. STB1 lama digunakan kembali; test tidak diekstrak.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import json, shutil, subprocess, sys, tarfile, time
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='agent/stb-capacity-causal-control'
if (REPO/'.git').is_dir():
 subprocess.run(['git','fetch','origin',BRANCH],cwd=REPO,check=True); subprocess.run(['git','checkout',BRANCH],cwd=REPO,check=True); subprocess.run(['git','reset','--hard',f'origin/{BRANCH}'],cwd=REPO,check=True)
else:
 if REPO.exists(): shutil.rmtree(REPO)
 command=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
 for attempt in range(1,4):
  result=subprocess.run(command)
  if result.returncode==0: break
  if REPO.exists(): shutil.rmtree(REPO)
  if attempt==3: raise RuntimeError('Git clone gagal tiga kali')
  time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
for key in list(sys.modules):
 if key=='coffee_detector' or key.startswith('coffee_detector.'): sys.modules.pop(key,None)
sys.path.insert(0,str(REPO/'src'))
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())

In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
assert torch.cuda.is_available(),'Aktifkan T4 GPU.'
REQUIRED=(
 'bundles/faruq-development-v3-grouped.tar',
 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
 'experiments/faruq-v3-breadth-screening-batch-v1/candidates/STB1/val_reports/stb_seed42_screening.json',)
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=REQUIRED)
ARCHIVE=require_project_artifact(PROJECT_ROOT,REQUIRED[0]); D0=require_project_artifact(PROJECT_ROOT,REQUIRED[1]); STB_SUMMARY=require_project_artifact(PROJECT_ROOT,REQUIRED[2])
DATA_ROOT=Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT/'data.yaml').is_file():
 with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
GROUPED_SUMMARY=DATA_ROOT/'faruq_grouped_summary.json'; assert GROUPED_SUMMARY.is_file(); assert not (DATA_ROOT/'test').exists()
OUTPUT_ROOT=PROJECT_ROOT/'experiments/faruq-v3-stb-capacity-control-v1'; OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
COMMON=['--data-root',str(DATA_ROOT),'--grouped-summary',str(GROUPED_SUMMARY),'--d0-checkpoint',str(D0),'--stb-summary',str(STB_SUMMARY),'--output-root',str(OUTPUT_ROOT),'--seed','42','--device','0']
print('GPU:',torch.cuda.get_device_name(0)); print('PROJECT:',PROJECT_ROOT); print('OUTPUT:',OUTPUT_ROOT)

## 1. Static capacity/identity gate — tanpa training

In [ ]:
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_stb_capacity_control',*COMMON,'--stage','static']
subprocess.run(command,cwd=REPO,check=True)
static=json.loads((OUTPUT_ROOT/'static_audit.json').read_text()); print('MODELS:',static['models']); print('PARAMETER GAP:',static['parameter_relative_gap']); print('GATES:',static['gates']); print('DECISION:',static['decision']); assert static['decision']=='PASS'

## 2. Train CMC0 saja — maksimal 50 epoch, resume langsung dari Drive

In [ ]:
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_stb_capacity_control',*COMMON,'--stage','train','--authorize-training']
print('MENJALANKAN:',' '.join(command),flush=True); subprocess.run(command,cwd=REPO,check=True)

In [ ]:
import pandas as pd
from IPython.display import display
SUMMARY=OUTPUT_ROOT/'val_reports/stb_capacity_control_seed42_decision.json'; result=json.loads(SUMMARY.read_text())
display(pd.DataFrame([{'model':name,**metrics} for name,metrics in result['models'].items()]).style.format({c:'{:.2%}' for c in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')}))
print('COMPARISONS:',json.dumps(result['comparisons'],indent=2)); print('DECISION:',result['decision']); print('NEXT:',result['next_action']); print('TEST OPENED:',result['test_opened']); print('Kirim tabel dan keputusan. Jangan membuka test atau menjalankan seed tambahan.')